# 03. Modeling – Stroke Prediction

Notebook này thực hiện quá trình xây dựng, đánh giá và lựa chọn mô hình dự đoán đột quỵ.

Quy trình chính:
- Chuẩn bị dữ liệu train/test
- Đánh giá 6 mô hình bằng Stratified 5-Fold Cross-Validation
- Sử dụng SMOTE bên trong Pipeline để xử lý mất cân bằng dữ liệu
- Lựa chọn các mô hình dựa trên kết quả Cross-Validation
- Tuning Random Forest và XGBoost
- Lựa chọn mô hình cuối cùng
- Xác định Decision Threshold
- Đánh giá mô hình trên Test Set

## 1. Import thư viện

Các thư viện được sử dụng cho quá trình xây dựng, đánh giá và tuning mô hình.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from src.models import get_models
from src.tuning import get_tuning_models, get_tuning_params
from src.preprocessing import RANDOM_STATE, get_preprocessor, load_data, split_data

print("Import OK")

Import OK


### Nhận xét

Các thư viện cần thiết cho quá trình xây dựng và đánh giá mô hình đã được import thành công.

Trong quá trình modeling, `Pipeline` được sử dụng để kết hợp tiền xử lý dữ liệu, SMOTE và mô hình. Điều này giúp hạn chế Data Leakage trong quá trình Cross-Validation.

## 2. Chuẩn bị dữ liệu

Dữ liệu được load và chia thành tập huấn luyện (Training Set) và tập kiểm tra (Test Set).

Tập Test được giữ riêng và chỉ sử dụng ở bước đánh giá cuối cùng.

In [2]:
df = load_data()

X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4088, 10)
X_test: (1022, 10)
y_train: (4088,)
y_test: (1022,)


### Nhận xét

Dữ liệu được chia thành:

- Training Set: 4.088 mẫu
- Test Set: 1.022 mẫu
- Số lượng biến đầu vào: 10

Training Set được sử dụng cho Cross-Validation và tuning mô hình. Test Set được giữ riêng để đánh giá mô hình cuối cùng.

## 3. Thiết lập Cross-Validation

Sử dụng Stratified 5-Fold Cross-Validation để đánh giá mô hình trên Training Set.

Stratified K-Fold giúp duy trì tỷ lệ giữa hai lớp của biến mục tiêu trong các fold.

In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = [
    "recall",
    "precision",
    "f1",
    "roc_auc",
    "average_precision"
]

print("Cross-Validation: 5-Fold Stratified")
print("Metrics:", scoring)

Cross-Validation: 5-Fold Stratified
Metrics: ['recall', 'precision', 'f1', 'roc_auc', 'average_precision']


### Nhận xét

Mỗi mô hình được đánh giá bằng 5-fold Cross-Validation với các chỉ số:

- Recall
- Precision
- F1-score
- ROC-AUC
- Average Precision

Trong bài toán dự đoán đột quỵ, dữ liệu mất cân bằng nên Average Precision và Recall được quan tâm bên cạnh các chỉ số còn lại.

## 4. Đánh giá 6 mô hình bằng Cross-Validation

Các mô hình được đánh giá gồm:

1. Logistic Regression
2. Decision Tree
3. KNN
4. SVM
5. Random Forest
6. XGBoost

Mỗi mô hình được đặt trong Pipeline gồm:

**Preprocessing → SMOTE → Model**

SMOTE được thực hiện bên trong Pipeline để tránh Data Leakage giữa các fold.

In [4]:
rows = []

for name, model in get_models().items():
    pipe = Pipeline([
        ("preprocessor", get_preprocessor()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

    res = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    row = {"model": name}

    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()

    rows.append(row)

    print("Xong:", name)

cv_results = pd.DataFrame(rows).round(3)

cv_results

Xong: Logistic Regression


Xong: Decision Tree


Xong: KNN


Xong: SVM


Xong: Random Forest


Xong: XGBoost


,model,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Logistic Regression,0.779,0.039,0.140,0.010,0.237,0.015,0.848,0.010,0.226,0.041
1,Decision Tree,0.221,0.018,0.143,0.021,0.173,0.020,0.576,0.011,0.070,0.006
2,KNN,0.412,0.054,0.127,0.011,0.194,0.018,0.684,0.039,0.100,0.009
3,SVM,0.483,0.082,0.122,0.016,0.194,0.025,0.787,0.025,0.147,0.030
4,Random Forest,0.121,0.009,0.202,0.021,0.150,0.010,0.803,0.016,0.144,0.011
5,XGBoost,0.110,0.060,0.187,0.114,0.138,0.078,0.801,0.018,0.151,0.028


### Nhận xét

Kết quả Cross-Validation cho thấy các mô hình có hiệu quả khác nhau trên dữ liệu mất cân bằng.

Logistic Regression đạt Average Precision cao nhất trong 6 mô hình với giá trị 0.226, đồng thời có ROC-AUC = 0.848 và Recall = 0.779.

XGBoost đạt Average Precision = 0.151 và ROC-AUC = 0.801.

Các kết quả này được sử dụng làm cơ sở để lựa chọn mô hình cho các bước tiếp theo.

In [5]:
cv_results.to_csv(RESULTS_DIR / "cv_results.csv", index=False)

print("Đã lưu kết quả vào: ../results/cv_results.csv")

Đã lưu kết quả vào: ../results/cv_results.csv


### Nhận xét

Kết quả Cross-Validation được lưu vào `results/cv_results.csv` để sử dụng cho báo cáo và các bước tiếp theo.

## 5. Lựa chọn 2 mô hình từ kết quả Cross-Validation

Dựa trên kết quả Cross-Validation, Average Precision được sử dụng làm tiêu chí chính để lựa chọn 2 mô hình.

Average Precision phù hợp với bài toán dữ liệu mất cân bằng vì đánh giá chất lượng dự đoán đối với lớp dương thông qua Precision-Recall.

In [6]:
# Đọc lại kết quả CV đã lưu
cv_results = pd.read_csv(RESULTS_DIR / "cv_results.csv")

# Chọn 2 mô hình tốt nhất theo Average Precision
top_2_models = (
    cv_results
    .sort_values("average_precision_mean", ascending=False)
    .head(2)
)

top_2_models[[
    "model",
    "average_precision_mean",
    "roc_auc_mean",
    "f1_mean",
    "recall_mean",
    "precision_mean"
]]

,model,average_precision_mean,roc_auc_mean,f1_mean,recall_mean,precision_mean
0,Logistic Regression,0.226,0.848,0.237,0.779,0.140
5,XGBoost,0.151,0.801,0.138,0.110,0.187


In [7]:
print("2 mô hình được chọn theo Average Precision:")
for _, row in top_2_models.iterrows():
    print(f"- {row['model']}: AP = {row['average_precision_mean']:.3f}")

2 mô hình được chọn theo Average Precision:
- Logistic Regression: AP = 0.226
- XGBoost: AP = 0.151


### Kết quả lựa chọn

Hai mô hình có Average Precision cao nhất là:

1. Logistic Regression — AP = 0.226
2. XGBoost — AP = 0.151

Đây là kết quả lựa chọn dựa trên Cross-Validation, không phải kết quả đánh giá trên Test Set.

## 6. Hyperparameter Tuning

Sau khi đánh giá các mô hình bằng Cross-Validation, bước tiếp theo là tối ưu siêu tham số cho Random Forest và XGBoost.

Quá trình tuning sử dụng RandomizedSearchCV với:
- Stratified 5-Fold Cross-Validation
- SMOTE được đặt bên trong Pipeline
- Average Precision làm tiêu chí đánh giá
- Chỉ sử dụng Training Set, không sử dụng Test Set

### 6.1. Tuning Random Forest

Random Forest được tối ưu các siêu tham số như số lượng cây, độ sâu cây, số mẫu tối thiểu để tách nhánh, số mẫu tối thiểu tại lá và số lượng đặc trưng được sử dụng tại mỗi lần chia.

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

from src.tuning import get_tuning_models, get_tuning_params

In [9]:
rf_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["Random Forest"]),
])

print("Random Forest pipeline OK")

Random Forest pipeline OK


In [10]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=get_tuning_params()["Random Forest"],
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("Random Forest RandomizedSearchCV OK")

Random Forest RandomizedSearchCV OK


In [11]:
rf_search.fit(X_train, y_train)

print("Random Forest tuning hoàn tất.")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Random Forest tuning hoàn tất.


In [12]:
print("Best CV Average Precision:", rf_search.best_score_)

Best CV Average Precision: 0.17691473481476175


In [13]:
print("Best Parameters:")
print(rf_search.best_params_)

Best Parameters:
{'model__max_depth': 5, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__n_estimators': 290}


### Nhận xét

Random Forest đã được tối ưu bằng RandomizedSearchCV với 20 bộ tham số và 5-fold Cross-Validation.

Bộ tham số tốt nhất được xác định gồm:
- `n_estimators = 290`
- `max_depth = 5`
- `max_features = log2`
- `min_samples_split = 6`
- `min_samples_leaf = 2`

Best CV Average Precision của Random Forest được sử dụng để so sánh với XGBoost trong bước tuning tiếp theo.

### 6.2. Tuning XGBoost

XGBoost được tối ưu bằng RandomizedSearchCV với cùng quy trình Cross-Validation như Random Forest.

Pipeline gồm:
- Tiền xử lý dữ liệu
- SMOTE để xử lý mất cân bằng lớp
- Mô hình XGBoost

Average Precision được sử dụng làm tiêu chí đánh giá trong quá trình tuning.

In [14]:
xgb_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["XGBoost"]),
])

print("XGBoost pipeline OK")

XGBoost pipeline OK


In [15]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=get_tuning_params()["XGBoost"],
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

print("XGBoost RandomizedSearchCV OK")

XGBoost RandomizedSearchCV OK


In [16]:
xgb_search.fit(X_train, y_train)

print("XGBoost tuning hoàn tất.")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


XGBoost tuning hoàn tất.


In [17]:
print("Best CV Average Precision:", xgb_search.best_score_)

Best CV Average Precision: 0.2138748015195079


In [18]:
print("Best Parameters:")
print(xgb_search.best_params_)

Best Parameters:
{'model__colsample_bytree': 0.7, 'model__gamma': 0.1, 'model__learning_rate': 0.03, 'model__max_depth': 2, 'model__min_child_weight': 4, 'model__n_estimators': 234, 'model__subsample': 0.7}


### Nhận xét

XGBoost được tối ưu bằng RandomizedSearchCV với 20 bộ tham số và 5-fold Cross-Validation.

Bộ tham số tốt nhất được xác định gồm:
- `n_estimators = 234`
- `max_depth = 2`
- `learning_rate = 0.03`
- `subsample = 0.7`
- `colsample_bytree = 0.7`
- `min_child_weight = 4`
- `gamma = 0.1`

Best CV Average Precision của XGBoost đạt **0.2139**. Kết quả này được đối chiếu với AP CV của Logistic Regression trước khi chốt mô hình.

In [19]:
tuning_results = pd.DataFrame({
    "model": ["Random Forest", "XGBoost"],
    "best_cv_average_precision": [
        rf_search.best_score_,
        xgb_search.best_score_
    ]
})

tuning_results

,model,best_cv_average_precision
0,Random Forest,0.176915
1,XGBoost,0.213875


### Nhận xét

Kết quả tuning cho thấy:

- Random Forest đạt Best CV Average Precision = **0.1769**.
- XGBoost đạt Best CV Average Precision = **0.2139**.

Logistic Regression có AP CV 0.226, cao hơn AP tuning của XGBoost (0.214) và Random Forest (0.177). Nhóm chọn Logistic Regression theo AP CV.

Test Set được giữ riêng trong toàn bộ quá trình lựa chọn và tuning.

In [20]:
tuning_results.to_csv(RESULTS_DIR / "tuning_results.csv", index=False)

print("Đã lưu results/tuning_results.csv")

Đã lưu results/tuning_results.csv


## Kết quả mô hình cuối và đánh giá holdout

Logistic Regression được chọn theo AP CV. Threshold 0.74 được khóa bằng F1 tối đa trên dự đoán out-of-fold của training set trước khi đánh giá holdout. Tập test đạt Recall 0.640, Precision 0.219, F1 0.327, ROC-AUC 0.845 và AP 0.275. Confusion matrix: TN=858, FP=114, FN=18, TP=32.

Pipeline cuối và các biểu đồ được tạo bằng `scripts/evaluate_final_model.py`. Cell sau đọc lại kết quả đã lưu, không fit hoặc đánh giá test lần nữa.

In [ ]:
from IPython.display import Image, display
import json

final_results = pd.read_csv(RESULTS_DIR / 'test_results.csv')
final_metadata = json.loads((RESULTS_DIR / 'final_model_metadata.json').read_text(encoding='utf-8'))
threshold_results = pd.read_csv(RESULTS_DIR / 'threshold_results.csv')
feature_importance = pd.read_csv(RESULTS_DIR / 'feature_importance.csv')
permutation_importance = pd.read_csv(RESULTS_DIR / 'permutation_importance.csv')
display(final_results)
display(threshold_results.loc[threshold_results['threshold'].sub(final_metadata['threshold']).abs().lt(1e-9)])
display(feature_importance.head(10))
display(permutation_importance.head(10))
for filename in ['confusion_matrix.png', 'roc_curve.png', 'pr_curve.png', 'threshold_metrics.png', 'feature_importance.png', 'permutation_importance.png']:
    display(Image(filename=str(RESULTS_DIR / filename)))